In [15]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV,StratifiedKFold
import pandas as pd
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn import tree
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models.signature import infer_signature
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
from sklearn.model_selection import RandomizedSearchCV
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../Data Files/Processed Files/credit_card_fraud_clean.csv')
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 2026, stratify = y)
sm = SMOTE(random_state = 2026)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [3]:
param_grid_rf = {
    "n_estimators": [200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "class_weight": ["balanced", "balanced_subsample"]
}

param_grid_hgb = {
    "learning_rate": [0.01, 0.05, 0.1],
    "max_iter": [100, 200],
    "max_depth": [5, 10],
    "min_samples_leaf": [20, 50],
    "l2_regularization": [0, 0.1, 1]
}

param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "scale_pos_weight": [5, 10, 20]
}

In [4]:
xgb_model = XGBClassifier(random_state=2026, use_label_encoder=False, eval_metric='logloss')
rf_model = RandomForestClassifier(random_state=2026)
hgb_model = HistGradientBoostingClassifier(random_state=2026)

In [5]:
search_xgb = RandomizedSearchCV(xgb_model, param_grid_xgb, n_iter=10, cv=StratifiedKFold(n_splits=5), scoring='recall', random_state=2026, n_jobs=-1, verbose=1)
search_rf = RandomizedSearchCV(rf_model, param_grid_rf, n_iter=10, cv=StratifiedKFold(n_splits=5), scoring='recall', random_state=2026, n_jobs=-1, verbose=1)
search_hgb = RandomizedSearchCV(hgb_model, param_grid_hgb, n_iter=10, cv=StratifiedKFold(n_splits=5), scoring='recall', random_state=2026, n_jobs=-1, verbose=1)

In [6]:
search_xgb.fit(X_train_res, y_train_res)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='logloss',
                                           feature_types=None,
                                           feature_weights=None, gamma=N...
                                           min_child_weight=None, missing=nan,
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.8, 1.0],
                                        'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [4, 6, 8],
                                        'n_estimators': [100, 200],
                                        'scale_pos_weight': [5, 10, 20],
                                        'subsample': [0.8, 1.0]},
                   random_state=2026, scoring='recall', verbose=1)

In [ ]:
search_rf.fit(X_train_res, y_train_res)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [7]:
search_hgb.fit(X_train_res, y_train_res)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                   estimator=HistGradientBoostingClassifier(random_state=2026),
                   n_jobs=-1,
                   param_distributions={'l2_regularization': [0, 0.1, 1],
                                        'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [5, 10],
                                        'max_iter': [100, 200],
                                        'min_samples_leaf': [20, 50]},
                   random_state=2026, scoring='recall', verbose=1)

In [8]:
xgb_best = search_xgb.best_estimator_
# rf_best = search_rf.best_estimator_
hgb_best = search_hgb.best_estimator_
models = [xgb_best, hgb_best]

In [13]:
model_metrics = {}
for model in models:
    model_name = type(model).__name__
    mlflow.set_experiment("Fraud Detection Fine Tuning")
    with mlflow.start_run(run_name=f"{model_name} Hyperparameter Tuning"):
        model.fit(X_train_res, y_train_res)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)
        conf_matrix = confusion_matrix(y_test, y_pred)
        
        precision = report['1']['precision']
        recall = report['1']['recall']
        f1_score = report['1']['f1-score']
        accuracy = accuracy_score(y_test, y_pred)
        
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1_score)
        signature = infer_signature(X_train_res, model.predict(X_train_res))
        if model_name == "XGBClassifier":
            mlflow.xgboost.log_model(model, artifact_path="model", signature=signature)
        else:
            mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)
        model_metrics[model_name] = {
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score
        }

2026/05/15 00:04:03 INFO mlflow.tracking.fluent: Experiment with name 'Fraud Detection Fine Tuning' does not exist. Creating a new experiment.


2026/05/15 00:04:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/15 00:04:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [14]:
model_metrics_df = pd.DataFrame(model_metrics).T
print(model_metrics_df)

                                accuracy  precision    recall  f1_score
XGBClassifier                   0.826080   0.028557  0.974719  0.055489
HistGradientBoostingClassifier  0.997365   0.699774  0.870787  0.775970


In [16]:
joblib.dump(hgb_best, '../Models/hgb_best_v1.pkl')

['../Models/hgb_best_v1.pkl']

In [4]:
from pathlib import Path
import glob

folder = Path("../App")

pkl_file = next(folder.glob("*.pkl"))

print(pkl_file.name)

hgb_best_v1.pkl
